In [9]:
### System Requirement ###
# >= 32 GB RAM (CPU)

# Set up the data for `playground_text.ipynb` (text-encoder decomposition)

Decomposes the CLIP **text** encoder at the EOS token into per-head attention and per-layer
MLP contributions (projected to the shared CLIP space), then derives the spectral (PC) text
explanations. The image-side probes (ImageNet embeddings + dataset) come from the original
`prepare_data.ipynb` and are reused for the bidirectional (image <-> text) view.

In [ ]:
## Parameters
device = 'cpu'
model_name = 'ViT-B-32'        # 'ViT-H-14', 'ViT-L-14', 'ViT-B-16', 'ViT-B-32'
seed = 0
num_last_layers_ = 4           # how many of the last TEXT-encoder layers to decompose
algorithm = "svd_data_approx"
# Text set decomposed into components (the corpus run through the text encoder, EOS analyzed).
dataset = "imagenet_descriptions_personal"
# Text set used to label the internal components.
dataset_text_name = "top_1500_nouns_5_sentences_imagenet_clean"
# imagenet_descriptions_personal has 10 "An image of ..." sentences per class, in class
# order; the LAST sentence of each block is the concise class-name-like summary.
native_per_class = 10          # sentences per class in `dataset` (1 = no class structure)
sentences_per_class = 1        # keep the LAST k per class; 1 -> only the class-name sentence
# Image set used to label the internal components  visualization.
datataset_image_name = "imagenet"
subset_dim = 10
tot_samples_per_class = 50
path = './datasets/'
cache_dir = "../cache"

if model_name == "ViT-H-14":
    pretrained = "laion2B-s32B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14":
    pretrained = "laion2B-s32B-b82K"; precision = "fp32"
elif model_name == "ViT-B-16":
    pretrained = "laion2B-s34B-b88K"; precision = "fp32"
elif model_name == "ViT-B-32":
    pretrained = "laion2B-s34B-b79K"; precision = "fp32"
elif model_name == "ViT-L-14-336":
    pretrained = "openai"; precision = "fp16"


Decompose the text dataset into per-head EOS components (i.e. the value added to the residual stream)
and optimized on by CLIP loss during training, and infered on.
Retrieve the activations (hooks) for each head, MLP in each layer, of a given text encoder model.
The output is 3 files, with the attn output (N, l, h, d), the mlp outputs (N, l, d), and the labels (N, )
as ordered index (from 0 to N - 1)

`{dataset}_attn_text_{model}_seed_{seed}.npy` `[N,l,h,d]`, `{dataset}_mlp_text_...` `[N,l+1,d]`.

In [ ]:
!python -m utils.scripts.compute_activation_values_text --device {device} --model {model_name} --pretrained {pretrained} --text_descriptions {dataset} --native_per_class {native_per_class} --sentences_per_class {sentences_per_class} --seed {seed} --cache_dir {cache_dir}

Using local files
Model parameters: 151,277,313
Context length: 77
Number of text layers: 12
Decomposing 14686 texts from 'top_1500_nouns_5_sentences_imagenet_clean'.
100%|██████████████████████████████████████| 58/58 [16:48:43<00:00, 1043.51s/it]

Concatenating chunk files into final .npy arrays...
Final single-file arrays created:
  ./output_dir/top_1500_nouns_5_sentences_imagenet_clean_attn_text_ViT-B-32_seed_0.npy  shape (14686, 12, 8, 512)
  ./output_dir/top_1500_nouns_5_sentences_imagenet_clean_mlp_text_ViT-B-32_seed_0.npy  shape (14686, 13, 512)
  ./output_dir/top_1500_nouns_5_sentences_imagenet_clean_labels_text_ViT-B-32_seed_0.npy  shape (14686,)
Chunk files removed. Done.


Compute the CLIP embeddings of the **descriptive/label** sentences (`dataset_text_name`) used to name the components and as text-side probes. Skip if `output_dir/{dataset_text_name}_{model}.npy` already exists.

In [ ]:
!python -m utils.scripts.compute_text_embeddings --device {device}  --model {model_name} --pretrained {pretrained} --data_path utils/text_descriptions/{dataset_text_name}.txt --cache_dir {cache_dir}

Using local files
Model parameters: 151,277,313
Context length: 77
Vocab size: 49408
/home/ggil/anaconda3/envs/MT/lib/python3.10/site-packages/torch/amp/autocast_mode.py:250: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
100%|████████████████████████████████████████████| 8/8 [18:23<00:00, 137.92s/it]


Ensure `output_dir/{image}_embeddings_{model}_seed_{seed}.npy` and the ImageNet dataset
exist (produced by `prepare_data.ipynb`: `compute_activation_values` +
`compute_images_embedding` on `--dataset imagenet`). Nothing to run here if they do.

Compute the ImageNet **class-prompt classifier** (`{image}_classifier_{model}.npy`), used to measure zero-shot accuracy of the reconstructed text embedding (labels = ImageNet class = row // sentences_per_class).

In [ ]:
!python -m utils.scripts.compute_classes_embeddings --device {device} --model {model_name} --pretrained {pretrained} --dataset {datataset_image_name} --cache_dir {cache_dir}

## 4. Spectral (PC) text explanations of the text-encoder heads
`svd_data_approx` on the last `num_last_layers_` layers; writes the `_completeness_text_*.jsonl`.

In [10]:
command = f"python -m utils.scripts.compute_text_explanations_text --device {device} --model {model_name} --algorithm {algorithm} --seed {seed} --text_per_princ_comp 20 --num_of_last_layers {num_last_layers_} --dataset {dataset} --text_descriptions {dataset_text_name}"
!{command}

Number of text layers: 12, heads: 8
  0%|                                                     | 0/4 [00:00<?, ?it/s]
Layer [8], Head: 0

Layer [8], Head: 1

Layer [8], Head: 2

Layer [8], Head: 3

Layer [8], Head: 4

Layer [8], Head: 5

Layer [8], Head: 6

Layer [8], Head: 7
 25%|███████████▎                                 | 1/4 [00:06<00:19,  6.55s/it]
Layer [9], Head: 0

Layer [9], Head: 1

Layer [9], Head: 2

Layer [9], Head: 3

Layer [9], Head: 4

Layer [9], Head: 5

Layer [9], Head: 6

Layer [9], Head: 7
 50%|██████████████████████▌                      | 2/4 [00:12<00:12,  6.41s/it]
Layer [10], Head: 0

Layer [10], Head: 1

Layer [10], Head: 2

Layer [10], Head: 3

Layer [10], Head: 4

Layer [10], Head: 5

Layer [10], Head: 6

Layer [10], Head: 7
 75%|█████████████████████████████████▊           | 3/4 [00:18<00:06,  6.27s/it]
Layer [11], Head: 0

Layer [11], Head: 1

Layer [11], Head: 2

Layer [11], Head: 3

Layer [11], Head: 4

Layer [11], Head: 5

Layer [11], Head: 6

Layer [1

Print the generated components (the per-head PC text explanations).

In [ ]:
# Print the text-encoder components we have generated
import json

jsonl = (f"output_dir/{dataset}_completeness_text_{dataset_text_name}_"
         f"{model_name}_algo_{algorithm}_seed_{seed}.jsonl")
with open(jsonl, "r") as json_file:
    for line in json_file:
        entry = json.loads(line)          # one (layer, head) per line; head == -1 is the final output
        layer, head = entry["layer"], entry["head"]
        if head == -1:
            print("final entry keys:", list(entry.keys()))
        print(f"Rank (nr PCs): {len(entry['s'])}  |  Layer: {layer}, Head: {head}")
        for pc in entry["embeddings_sort"]:
            print(pc["text"])
